# Legal Document Summarization — Beginner-Friendly Version

This notebook does the same project as before (summarizing NDAs and SLAs),
but the code is written to be **easy to read and follow**, even if you're
new to Python or machine learning.

**What we'll do, step by step:**
1. Install the tools we need
2. Load some legal contracts (from a dataset called CUAD)
3. Clean and prepare the text
4. Try a simple "extractive" summarizer (pulls out important sentences)
5. Try an "abstractive" summarizer (writes a new summary in its own words)
6. (Optional) Fine-tune a model on our own data
7. Check how good our summaries are
8. Summarize a few contracts and look at the results

**How this notebook is written**
- Every code cell has comments (lines starting with `#`) explaining what
  that line does.
- We avoid fancy shortcuts and write things out in plain, simple steps.
- We use `print()` a lot so you can see what's happening at each step.
- Each cell does ONE thing, so if something breaks, you know exactly where.

Run the cells **in order, from top to bottom**.


## Step 1: Install and import the tools we need

We need a few Python libraries:
- `transformers` — gives us pre-trained AI models like BART and T5
- `datasets` — helps us load the CUAD legal contracts dataset
- `sumy` — a simple library for extractive summarization
- `evaluate` and `rouge_score` — help us score how good our summaries are

This step only needs to run once.


In [ ]:
# The "!" lets us run a terminal command from inside the notebook.
# "-q" just means "quiet" (less text printed while installing).

!pip install -q -U transformers datasets sumy evaluate rouge_score nltk scikit-learn


In [5]:
# Now we import (load) the libraries we just installed, plus some
# standard Python tools we'll use throughout the notebook.

import re            # for simple text pattern matching
import json          # for reading/writing JSON files
import textwrap       # for neatly printing long text
import pandas as pd   # for working with tables of data

# This tells us whether a GPU is available, which makes AI models run faster.
# It's fine if this says "cpu" — everything will still work, just slower.
import torch
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("We are using:", device)


We are using: cpu


## Step 2: Load some legal contracts

We'll use a public dataset called **CUAD** (Contract Understanding Atticus
Dataset). It contains real contracts with important clauses already
labeled — for example, it tells us which sentences are about
"Confidentiality" or "Indemnification".

CUAD does **not** come with ready-made summaries, only labeled clauses.
That's okay — we'll use those clause labels later to check whether our
summaries mention the important clauses.


In [ ]:
# The original approach used the `datasets` library's built-in CUAD loader,
# but that loader is an older-style Python "loading script", and recent
# versions of the `datasets` library refuse to run those for security
# reasons. You would see an error like:
#   RuntimeError: Dataset scripts are no longer supported
#
# FIX: instead of going through the `datasets` library, we download CUAD
# directly from its official GitHub repo. It ships as a zip file containing
# a SQuAD-style JSON file, which is the exact same format the old loader
# used to produce -- so everything below (which expects "title", "context",
# "question", "answers") still works with no other changes needed.
import io
import zipfile
import urllib.request

print("Downloading CUAD... this might take a minute the first time.")

CUAD_ZIP_URL = "https://github.com/TheAtticusProject/cuad/raw/main/data.zip"

with urllib.request.urlopen(CUAD_ZIP_URL) as response:
    zip_bytes = response.read()

# The zip contains a SQuAD-style JSON file with all the labeled clauses.
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    json_names = [name for name in zf.namelist() if name.endswith(".json")]
    with zf.open(json_names[0]) as f:
        cuad_raw = json.load(f)

# Flatten the SQuAD-style structure into a simple list of examples, one per
# (contract, clause-question) pair -- this mirrors what the old HF loader gave us.
cuad_train = []
for article in cuad_raw["data"]:
    title = article["title"]
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            cuad_train.append({
                "id": qa["id"],
                "title": title,
                "context": context,
                "question": qa["question"],
                "answers": {
                    "text": [a["text"] for a in qa["answers"]],
                    "answer_start": [a["answer_start"] for a in qa["answers"]],
                },
            })

print("Number of rows in CUAD:", len(cuad_train))
print()
print("Here is what ONE row looks like:")
print(cuad_train[0])


In [ ]:
# Each row in CUAD has:
#   "title"     -> the name of the contract
#   "context"   -> the full text of the contract
#   "question"  -> which clause type this row is about (e.g. "Confidentiality")
#   "answers"   -> the actual text of that clause, if it exists in the contract

# Let's turn this into a simple table (a pandas DataFrame) that's easier to work with.

rows = []

for example in cuad_train:
    title = example["title"]
    contract_text = example["context"]
    question = example["question"]
    answer_list = example["answers"]["text"]  # this is a list, could be empty

    # Only keep rows where there IS an answer (i.e. this clause exists in the contract)
    if len(answer_list) > 0:
        clause_text = answer_list[0]
        rows.append({
            "title": title,
            "contract_text": contract_text,
            "question": question,
            "clause_text": clause_text,
        })

cuad_df = pd.DataFrame(rows)
print("We now have", len(cuad_df), "labeled clauses.")
cuad_df.head()


In [ ]:
# The "question" column is a full sentence, like:
# "Highlight the parts (if any) of this contract related to Confidentiality"
# Let's turn that into a short, simple clause name like "Confidentiality".

# This is the list of clause types we care about most for NDAs and SLAs.
clause_names_we_want = [
    "Confidentiality",
    "Indemnification",
    "Limitation of Liability",
    "Termination",
    "Governing Law",
    "Auto-Renewal",
    "Warranty",
]

def get_simple_clause_name(question_text):
    # Check if any of our clause names appears inside the question text.
    for name in clause_names_we_want:
        if name.lower() in question_text.lower():
            return name
    # If none matched, just call it "Other"
    return "Other"

cuad_df["clause_name"] = cuad_df["question"].apply(get_simple_clause_name)

# Let's see how many of each clause type we found
print(cuad_df["clause_name"].value_counts())


In [ ]:
# Contracts are long, and each contract has MANY rows (one per clause type).
# Let's group everything by contract title, so we have ONE row per contract,
# with a list of all its labeled clauses attached.

contract_titles = cuad_df["title"].unique()
print("Number of unique contracts:", len(contract_titles))

contracts_list = []

for title in contract_titles:
    # Get all rows for this one contract
    rows_for_this_contract = cuad_df[cuad_df["title"] == title]

    # The full contract text is the same in every row, so just grab the first one
    full_text = rows_for_this_contract.iloc[0]["contract_text"]

    # Collect (clause_name, clause_text) pairs for this contract
    clause_pairs = []
    for _, r in rows_for_this_contract.iterrows():
        clause_pairs.append((r["clause_name"], r["clause_text"]))

    contracts_list.append({
        "title": title,
        "full_text": full_text,
        "clauses": clause_pairs,
    })

contracts_df = pd.DataFrame(contracts_list)
print("Rebuilt", len(contracts_df), "contracts, each with their labeled clauses.")
contracts_df.iloc[0]["clauses"][:3]


## Step 3: Prepare the text (chunking)

AI summarization models can only read a limited amount of text at once
(usually around 1,000 words). Legal contracts are often much longer than
that, so we need to **split them into smaller pieces (chunks)** before
summarizing.

To keep this simple, we'll split each contract into chunks of a fixed
number of words, making sure we don't cut off in a strange spot.


In [ ]:
def split_text_into_chunks(text, words_per_chunk=600):
    """
    Splits a long piece of text into smaller chunks.
    Each chunk has roughly 'words_per_chunk' words.
    This is a simple, beginner-friendly way to handle long documents.
    """
    words = text.split()  # break the text into a list of words
    chunks = []

    start = 0
    while start < len(words):
        end = start + words_per_chunk
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)
        chunks.append(chunk_text)
        start = end  # move to the next chunk

    return chunks

# Let's test this on the first contract
sample_contract = contracts_df.iloc[0]["full_text"]
sample_chunks = split_text_into_chunks(sample_contract)

print("The contract has about", len(sample_contract.split()), "words")
print("It was split into", len(sample_chunks), "chunk(s)")
print()
print("Here is the start of chunk 1:")
print(textwrap.fill(sample_chunks[0][:400], 100))


## Step 4: Extractive summarization (LexRank)

**Extractive summarization** means picking out the most important
*existing* sentences from the document — it doesn't write anything new.

We'll use a simple, well-known method called **LexRank**, provided by the
`sumy` library. This is a good, safe baseline because it never makes
anything up — every sentence in the summary really is in the contract.


In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

def summarize_extractive(text, number_of_sentences=5):
    """
    Picks the most important sentences from the text using LexRank.
    Returns them joined together as a summary.
    """
    parser = PlaintextParser.from_string(text, SumyTokenizer("english"))
    stemmer = Stemmer("english")

    summarizer = LexRankSummarizer(stemmer)
    summarizer.stop_words = get_stop_words("english")

    # This returns a list of the "best" sentences
    best_sentences = summarizer(parser.document, number_of_sentences)

    # Join the sentences into one summary string
    summary = " ".join(str(sentence) for sentence in best_sentences)
    return summary

# Try it on our sample contract
extractive_summary = summarize_extractive(sample_contract, number_of_sentences=5)

print("EXTRACTIVE SUMMARY:")
print(textwrap.fill(extractive_summary, 100))


## Step 5: Abstractive summarization (BART)

**Abstractive summarization** means the AI model *writes a new summary in
its own words*, instead of just copying sentences. We'll use a
pre-trained model called **BART**, which is already very good at
summarizing text — we don't need to train it ourselves to get started.

Because BART can only read about 1,000 words at a time, we'll:
1. Split the contract into chunks (using our function from Step 3)
2. Summarize each chunk separately
3. Combine the chunk summaries into one final summary


In [ ]:
# The old shortcut `pipeline("summarization", ...)` was REMOVED in
# Transformers v5 (along with the "translation" and "text2text-generation"
# pipeline tasks). Running it now raises:
#   KeyError: "Unknown task summarization, available tasks are [...]"
#
# FIX: load the tokenizer and model directly instead of going through
# pipeline(). This is the officially recommended replacement, and it works
# whether you have an old or new version of the `transformers` library.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading the BART summarization model...")
summarizer_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
summarizer_model.to(device)
print("Model loaded!")


In [ ]:
def summarize_abstractive(text, words_per_chunk=600):
    """
    Summarizes a long document using BART.
    Handles long documents by chunking, summarizing each chunk,
    then combining the results.
    """
    def run_bart_summary(input_text, max_length=120, min_length=20):
        # Turn the text into token IDs the model can read.
        inputs = summarizer_tokenizer(
            input_text,
            max_length=1024,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        # Ask the model to generate a summary (do_sample=False keeps the
        # output consistent each time we run it).
        summary_ids = summarizer_model.generate(
            **inputs,
            max_length=max_length,
            min_length=min_length,
            do_sample=False,
        )

        # Turn the generated token IDs back into readable text.
        return summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Step 1: split into chunks so BART can read them
    chunks = split_text_into_chunks(text, words_per_chunk)

    # Step 2: summarize each chunk one at a time
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Summarizing chunk {i + 1} of {len(chunks)}...")
        chunk_summaries.append(run_bart_summary(chunk, max_length=120, min_length=20))

    # Step 3: if there was only one chunk, that's already our summary
    if len(chunk_summaries) == 1:
        return chunk_summaries[0]

    # If there were multiple chunks, combine their summaries and
    # summarize ONE more time to make a single, shorter final summary.
    combined_text = " ".join(chunk_summaries)
    return run_bart_summary(combined_text, max_length=150, min_length=30)

# Try it on our sample contract
abstractive_summary = summarize_abstractive(sample_contract)

print()
print("ABSTRACTIVE SUMMARY:")
print(textwrap.fill(abstractive_summary, 100))


## Step 6 (Optional): Fine-tuning on your own data

Fine-tuning means "further training" a model on your own examples, so it
gets better at your specific task (legal summaries).

**This step needs real training data**: pairs of `(contract text, good
summary written by a lawyer)`. We don't have that yet, so this section
shows the pattern in simple steps, using a small **placeholder** dataset.
Swap in real lawyer-written summaries before trusting the output of this
step.


In [ ]:
# STEP 6a: Build a small training dataset.
# Normally you would load real (contract, lawyer-written summary) pairs here,
# for example from a CSV file. Since we don't have one yet, we build a tiny
# PLACEHOLDER dataset using the clause text CUAD already gave us.
# This is only so the training code below has something to run on.

training_examples = []

for _, contract_row in contracts_df.iterrows():
    for clause_name, clause_text in contract_row["clauses"]:
        # Skip very short clauses, they are not useful training examples
        if len(clause_text.split()) < 8:
            continue

        # PLACEHOLDER target: just shorten the clause text.
        # Replace this with a REAL summary written by a person.
        words = clause_text.split()
        placeholder_summary = " ".join(words[:30])

        training_examples.append({
            "source_text": clause_text,
            "target_summary": placeholder_summary,
        })

print("Built", len(training_examples), "placeholder training examples.")
print("Example:")
print(training_examples[0])


In [ ]:
# STEP 6b: Split into a training set and a small evaluation set.
from sklearn.model_selection import train_test_split

train_examples, eval_examples = train_test_split(
    training_examples, test_size=0.15, random_state=42
)

print("Training examples:", len(train_examples))
print("Evaluation examples:", len(eval_examples))


In [ ]:
# STEP 6c: Turn our examples into a Hugging Face Dataset,
# and load a tokenizer + model to fine-tune.

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"  # you could also try "t5-small" here

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

train_dataset = Dataset.from_list(train_examples)
eval_dataset = Dataset.from_list(eval_examples)

print("Model and tokenizer loaded.")


In [ ]:
# STEP 6d: Convert text into numbers the model can understand (tokenizing).

def tokenize_examples(batch):
    # Turn the source text into token IDs (numbers)
    model_inputs = tokenizer(
        batch["source_text"],
        max_length=512,
        truncation=True,
    )
    # Turn the target summary into token IDs too — this becomes the "label"
    labels = tokenizer(
        text_target=batch["target_summary"],
        max_length=100,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset_tokenized = train_dataset.map(tokenize_examples, batched=True)
eval_dataset_tokenized = eval_dataset.map(tokenize_examples, batched=True)

print("Tokenizing done.")


In [ ]:
# STEP 6e: Set up training and run it.
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_settings = Seq2SeqTrainingArguments(
    output_dir="/home/claude/my_finetuned_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,     # small batch size, easier on memory
    per_device_eval_batch_size=2,
    num_train_epochs=1,                # just 1 pass over the data to keep this quick
    predict_with_generate=True,
    logging_steps=10,
    report_to="none",
)

# NOTE: Transformers v5 removed the `tokenizer=` argument from Trainer
# (it used to just warn about this, now it raises a TypeError). Use
# `processing_class=` instead -- it does the same job.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_settings,
    train_dataset=train_dataset_tokenized,
    eval_dataset=eval_dataset_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# This actually starts training. It can take a while, especially on CPU.
trainer.train()


In [ ]:
# STEP 6f: Save the fine-tuned model so we can use it later.

trainer.save_model("/home/claude/my_finetuned_model")
tokenizer.save_pretrained("/home/claude/my_finetuned_model")

print("Fine-tuned model saved!")


## Step 7: Check how good our summaries are

We'll compare our summary to a "gold" (correct/reference) summary using a
few simple, well-known scoring methods:

- **ROUGE** — measures how many words/phrases overlap between our summary
  and the reference summary. Higher is better.
- **Clause coverage** — a simple check we write ourselves: does our
  summary mention the important clause types (Confidentiality,
  Indemnification, etc.)?


In [ ]:
import evaluate

rouge_scorer = evaluate.load("rouge")

def score_with_rouge(predicted_summary, reference_summary):
    """
    Compares our summary to a reference summary using ROUGE.
    Returns a dictionary of scores between 0 and 1 (higher = better).
    """
    scores = rouge_scorer.compute(
        predictions=[predicted_summary],
        references=[reference_summary],
    )
    return scores

# Example: compare our extractive and abstractive summaries to
# each other, just to see the function work. In Step 8 we will
# compare against real reference summaries.
example_scores = score_with_rouge(abstractive_summary, extractive_summary)
print(example_scores)


In [ ]:
def check_clause_coverage(summary_text, clause_pairs):
    """
    A simple, easy-to-understand check:
    for each clause type in the contract (e.g. "Confidentiality"),
    does the WORD appear somewhere in our summary?

    This is a simple word-matching check, not a perfect measure of meaning,
    but it is easy to understand and a good starting point.
    """
    # Get the unique list of clause names in this contract (ignore "Other")
    clause_names = []
    for name, _ in clause_pairs:
        if name != "Other" and name not in clause_names:
            clause_names.append(name)

    if len(clause_names) == 0:
        return None  # nothing to check

    covered = []
    missing = []

    for name in clause_names:
        if name.lower() in summary_text.lower():
            covered.append(name)
        else:
            missing.append(name)

    coverage_score = len(covered) / len(clause_names)

    print("Clause types in this contract:", clause_names)
    print("Mentioned in summary:", covered)
    print("Missing from summary:", missing)
    print("Coverage score:", round(coverage_score, 2))

    return coverage_score

# Try it on our abstractive summary
check_clause_coverage(abstractive_summary, contracts_df.iloc[0]["clauses"])


## Step 8: Summarize a few contracts and compare results

Now let's put it all together: pick a handful of contracts, summarize each
one with both methods (extractive and abstractive), and look at the
results side by side in a simple table.


In [ ]:
# Pick a small number of contracts to test on (10, or fewer if we don't have 10)
number_to_test = min(10, len(contracts_df))
test_contracts = contracts_df.sample(n=number_to_test, random_state=42).reset_index(drop=True)

print("We will summarize", number_to_test, "contracts.")


In [ ]:
# For each contract, we'll create both types of summaries, then store the results.
# NOTE: this may take a few minutes since abstractive summarization is slower.

results = []

for index, row in test_contracts.iterrows():
    title = row["title"]
    text = row["full_text"]
    clauses = row["clauses"]

    print("---")
    print(f"Contract {index + 1} of {number_to_test}: {title}")

    extractive_result = summarize_extractive(text, number_of_sentences=5)
    abstractive_result = summarize_abstractive(text)

    results.append({
        "title": title,
        "extractive_summary": extractive_result,
        "abstractive_summary": abstractive_result,
        "clauses": clauses,
    })

print("---")
print("Done summarizing all contracts!")


In [ ]:
# Let's look at one example in full, so we can read and compare the two summaries.

first_result = results[0]

print("CONTRACT:", first_result["title"])
print()
print("EXTRACTIVE SUMMARY:")
print(textwrap.fill(first_result["extractive_summary"], 100))
print()
print("ABSTRACTIVE SUMMARY:")
print(textwrap.fill(first_result["abstractive_summary"], 100))


In [ ]:
# Now let's check clause coverage for every contract and both summary types,
# and put everything into one easy-to-read table.

table_rows = []

for r in results:
    extractive_coverage = check_clause_coverage(r["extractive_summary"], r["clauses"])
    abstractive_coverage = check_clause_coverage(r["abstractive_summary"], r["clauses"])

    table_rows.append({
        "title": r["title"],
        "extractive_clause_coverage": extractive_coverage,
        "abstractive_clause_coverage": abstractive_coverage,
    })

coverage_table = pd.DataFrame(table_rows)
coverage_table


In [ ]:
# Finally, let's see the AVERAGE clause coverage across all contracts,
# for each summarization method. This gives us a simple way to compare them.

average_extractive = coverage_table["extractive_clause_coverage"].mean()
average_abstractive = coverage_table["abstractive_clause_coverage"].mean()

print("Average clause coverage:")
print("  Extractive (LexRank):", round(average_extractive, 2))
print("  Abstractive (BART):  ", round(average_abstractive, 2))


## What we learned

- **Extractive (LexRank)** summaries are made entirely of real sentences
  from the contract — safer, but can read a bit choppy.
- **Abstractive (BART)** summaries are written in new words — easier to
  read, but there's a small risk it changes the meaning slightly
  (this is why we check "clause coverage" and would want a lawyer to
  double check important results).
- **Clause coverage** tells us whether the important legal topics
  (Confidentiality, Indemnification, etc.) actually show up in the
  summary — this matters more than just "sounding good."

### Where to go from here
- Replace the placeholder training data in Step 6 with real, lawyer-written
  summaries.
- Try `t5-small` or `t5-base` as an alternative model and compare results.
- Add ROUGE scoring in Step 8 once you have real reference summaries to
  compare against.
